# 02 — Classify direct appeals to Commission decisions

This notebook classifies the relationship between EU Court documents and the Commission decision identifiers detected in `01_court_decision_regex_v2`.

The unit of classification is **one Court document × one normalized Commission identifier**. Repeated regex hits for the same identifier in the same document are therefore considered together rather than classified separately.

The main question is whether the present Court proceeding is a **direct judicial challenge to the identified Commission decision**. If it is not, the notebook assigns a broad citation role describing how the Commission decision is used in the Court document.

This notebook does not search for additional Commission decisions. Candidate discovery belongs to notebook 01; this notebook only classifies the candidates already identified there.

## Inputs and outputs

**Input**

- `output/court_regex/court_decision_regex.xlsx`
- sheet `references`, produced by `01_court_decision_regex_v2`

Only the two Commission-decision identifier families used for relationship classification are retained:

- `decision_case_number`
- `decision_oj_reference`

**Prepared model inputs**

Before classification begins, the notebook creates one persistent UTF-8 text file per aggregated candidate under:

- `logs/court_regex/direct_appeal_inputs/`

For shorter Court documents, the complete text is retained. For longer documents, the prepared input contains:

- the first 15,000 characters;
- a ±2,000-character context around every regex hit for that particular candidate, with overlapping windows merged;
- the final 10,000 characters.

The prepared text is cleaned of problematic control characters before it is saved. Classification reads these saved files rather than rebuilding snippets during the model call.

**Runtime log**

- `logs/court_regex/direct_appeals_classification.jsonl`

The JSONL file is an append-only checkpoint. Successful jobs are skipped when the notebook is rerun, so classification can be stopped and resumed.

**Final output**

- `output/court_regex/court_direct_appeals.xlsx`

The main sheet contains the original aggregated regex candidate together with the direct-appeal classification and citation role.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import asyncio
import hashlib
import json
import os
import re

import httpx
import pandas as pd
from tqdm.auto import tqdm

## 1. Paths and settings

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root

    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "code" / "03_matching").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / "output").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Run the notebook from within the "
        "repository or set ECCJEU_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()

REGEX_XLSX = PROJECT_ROOT / "output" / "court_regex" / "court_decision_regex.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "court_regex"
LOG_DIR = PROJECT_ROOT / "logs" / "court_regex"
INPUT_TEXT_DIR = LOG_DIR / "direct_appeal_inputs"

CHECKPOINT_JSONL = LOG_DIR / "direct_appeals_classification.jsonl"
INPUT_MANIFEST_CSV = LOG_DIR / "direct_appeal_input_manifest.csv"
OUTPUT_XLSX = OUTPUT_DIR / "court_direct_appeals.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
INPUT_TEXT_DIR.mkdir(parents=True, exist_ok=True)

# Prepared-input strategy
FULL_DOCUMENT_MAX_CHARS = 120_000
BEGINNING_CHARS = 15_000
ENDING_CHARS = 10_000
HIT_CONTEXT_CHARS = 2_000

# Local vLLM / OpenAI-compatible endpoint
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "EMPTY")
MODEL = os.getenv("VLLM_MODEL", "google/gemma-4-31B-it")

# Start conservatively. Increase to 6 or 8 after confirming the local server is stable.
MAX_WORKERS = int(os.getenv("DIRECT_APPEAL_WORKERS", "8"))
REQUEST_TIMEOUT_SECONDS = 3600
MAX_RETRIES = 2
TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 1800

# "update" resumes from successful JSONL records.
# "redo_all" ignores the existing checkpoint and reruns every candidate.
# "rebuild_excel" makes no model calls and only rebuilds the workbook.
RUN_MODE = os.getenv("RUN_MODE", "update").strip().lower()
if RUN_MODE not in {"update", "redo_all", "rebuild_excel"}:
    raise ValueError("RUN_MODE must be update, redo_all, or rebuild_excel")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Regex input:", REGEX_XLSX)
print("Prepared input directory:", INPUT_TEXT_DIR)
print("Checkpoint:", CHECKPOINT_JSONL)
print("Output:", OUTPUT_XLSX)
print("Workers:", MAX_WORKERS)
print("Run mode:", RUN_MODE)

## 2. Load the aggregated Commission-decision candidates

Notebook 01 already groups repeated mentions of the same normalized identifier within a Court document and stores the position of every underlying regex hit in `hit_spans_json`.

Those grouped rows are the classification jobs used here. CELEX hits are not classified separately because the present task focuses on Commission case numbers and published decision references.

In [ ]:
if not REGEX_XLSX.exists():
    raise FileNotFoundError(
        f"Required regex workbook is missing: {REGEX_XLSX}\n"
        "Run 01_court_decision_regex_v2 first."
    )

references = pd.read_excel(
    REGEX_XLSX,
    sheet_name="references",
    dtype=str,
    keep_default_na=False,
)
references.columns = [str(c).strip() for c in references.columns]

required_columns = {
    "record_id",
    "affId",
    "procedureId",
    "case_number_clean",
    "clean_file_path",
    "reference_type",
    "reference_normalized",
    "reference_count",
    "hit_spans_json",
}
missing = sorted(required_columns - set(references.columns))
if missing:
    raise ValueError(f"Regex references sheet is missing required columns: {missing}")

CLASSIFIED_REFERENCE_TYPES = {"decision_case_number", "decision_oj_reference"}

jobs_df = references.loc[
    references["reference_type"].isin(CLASSIFIED_REFERENCE_TYPES)
].copy()

jobs_df = jobs_df.reset_index(drop=True)

def stable_job_id(row: pd.Series) -> str:
    key = "|".join(
        [
            str(row.get("record_id", "")),
            str(row.get("reference_type", "")),
            str(row.get("reference_normalized", "")),
        ]
    )
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]

jobs_df["job_id"] = jobs_df.apply(stable_job_id, axis=1)

if jobs_df["job_id"].duplicated().any():
    dupes = jobs_df.loc[jobs_df["job_id"].duplicated(keep=False)]
    raise ValueError(
        "Duplicate classification job IDs were produced. Inspect:\n"
        + dupes[["record_id", "reference_type", "reference_normalized"]]
        .head(20)
        .to_string(index=False)
    )

print(f"Aggregated Commission-decision candidates: {len(jobs_df):,}")
display(
    jobs_df["reference_type"]
    .value_counts()
    .rename_axis("reference_type")
    .reset_index(name="candidates")
)
display(jobs_df.head(20))

## 3. Prepare and save the exact model inputs

The model inputs are prepared **before any classification calls are made**.

For each aggregated candidate, the stored hit positions from notebook 01 are used to recover every mention of that candidate in the clean Court text. For long documents, overlapping context windows are merged so that nearby mentions are not duplicated.

The resulting text files are persistent records of what the classifier actually saw. They can therefore be inspected independently of the model run.

In [ ]:
ILLEGAL_MODEL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")

def resolve_repo_path(value: str) -> Path | None:
    raw = str(value or "").strip()
    if not raw:
        return None

    p = Path(raw).expanduser()
    if p.is_absolute():
        return p.resolve()

    return (PROJECT_ROOT / p).resolve()


def clean_model_text(text: str) -> str:
    """Remove problematic controls while preserving ordinary line structure."""
    text = str(text or "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = ILLEGAL_MODEL_CHARS_RE.sub("", text)

    # Replace unpaired surrogates or other strings that cannot be written as UTF-8.
    return text.encode("utf-8", errors="replace").decode("utf-8")


def parse_hit_spans(value: str) -> list[tuple[int, int]]:
    try:
        raw = json.loads(str(value or "[]"))
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid hit_spans_json: {value!r}") from exc

    spans: list[tuple[int, int]] = []
    for item in raw:
        if not isinstance(item, list) or len(item) != 2:
            raise ValueError(f"Invalid hit span: {item!r}")
        start, end = int(item[0]), int(item[1])
        if start < 0 or end < start:
            raise ValueError(f"Invalid hit span boundaries: {item!r}")
        spans.append((start, end))

    return spans


def merge_intervals(
    intervals: list[tuple[int, int]],
    *,
    text_length: int,
) -> list[tuple[int, int]]:
    clipped = sorted(
        (
            max(0, int(start)),
            min(text_length, int(end)),
        )
        for start, end in intervals
        if int(end) > 0 and int(start) < text_length
    )

    if not clipped:
        return []

    merged: list[list[int]] = [[clipped[0][0], clipped[0][1]]]
    for start, end in clipped[1:]:
        last = merged[-1]
        if start <= last[1]:
            last[1] = max(last[1], end)
        else:
            merged.append([start, end])

    return [(start, end) for start, end in merged]


def safe_filename_part(value: str, max_len: int = 60) -> str:
    value = str(value or "").strip()
    value = re.sub(r"\s+", "-", value)
    value = re.sub(r"[^A-Za-z0-9._-]+", "-", value)
    value = re.sub(r"-+", "-", value).strip("-._")
    return (value or "unknown")[:max_len]


def prepared_input_filename(row: pd.Series) -> str:
    source_case = (
        str(row.get("case_number_clean", "")).strip()
        or str(row.get("affId", "")).strip()
        or str(row.get("record_id", "")).strip()
    )
    return (
        f"{safe_filename_part(row['reference_type'])}"
        f"__{safe_filename_part(source_case)}"
        f"__{safe_filename_part(row['reference_normalized'])}"
        f"__{row['job_id']}.txt"
    )


def build_prepared_input(
    text: str,
    *,
    hit_spans: list[tuple[int, int]],
    reference_normalized: str,
    reference_type: str,
) -> tuple[str, str, list[tuple[int, int]]]:
    text_length = len(text)

    for start, end in hit_spans:
        if start > text_length or end > text_length:
            raise ValueError(
                f"Stored regex span [{start}, {end}] exceeds source-text length "
                f"{text_length:,}. Rerun notebook 01 if the clean text changed."
            )

    if text_length <= FULL_DOCUMENT_MAX_CHARS:
        header = (
            f"REFERENCE TYPE: {reference_type}\n"
            f"TARGET COMMISSION IDENTIFIER: {reference_normalized}\n"
            f"INPUT STRATEGY: full_document\n"
            f"REGEX HITS IN DOCUMENT: {len(hit_spans)}\n\n"
            "===== FULL COURT DOCUMENT =====\n"
        )
        return clean_model_text(header + text), "full_document", [(0, text_length)]

    intervals: list[tuple[int, int]] = [
        (0, min(BEGINNING_CHARS, text_length)),
        (max(0, text_length - ENDING_CHARS), text_length),
    ]

    for start, end in hit_spans:
        intervals.append(
            (
                max(0, start - HIT_CONTEXT_CHARS),
                min(text_length, end + HIT_CONTEXT_CHARS),
            )
        )

    merged = merge_intervals(intervals, text_length=text_length)

    sections: list[str] = [
        f"REFERENCE TYPE: {reference_type}",
        f"TARGET COMMISSION IDENTIFIER: {reference_normalized}",
        "INPUT STRATEGY: focused_document",
        f"ORIGINAL DOCUMENT CHARACTERS: {text_length}",
        f"REGEX HITS IN DOCUMENT: {len(hit_spans)}",
        "",
        (
            "The sections below are extracts from one Court document. They include "
            "the beginning, the end, and context around every regex occurrence of "
            "the target Commission identifier. Overlapping sections have been merged."
        ),
    ]

    for index, (start, end) in enumerate(merged, start=1):
        sections.extend(
            [
                "",
                f"===== SOURCE CHARACTERS {start}:{end} | SECTION {index}/{len(merged)} =====",
                text[start:end],
            ]
        )

    return clean_model_text("\n".join(sections)), "focused_document", merged


prepared_rows: list[dict[str, Any]] = []

for _, row in tqdm(
    jobs_df.iterrows(),
    total=len(jobs_df),
    desc="Preparing candidate texts",
):
    source_path = resolve_repo_path(row["clean_file_path"])
    preparation_status = "ready"
    preparation_error = ""
    original_chars = 0
    prepared_chars = 0
    input_strategy = ""
    merged_sections: list[tuple[int, int]] = []
    input_path = INPUT_TEXT_DIR / prepared_input_filename(row)

    try:
        if source_path is None or not source_path.exists():
            raise FileNotFoundError(f"Clean Court text is missing: {source_path}")

        source_text = source_path.read_text(encoding="utf-8", errors="replace")
        original_chars = len(source_text)

        hit_spans = parse_hit_spans(row["hit_spans_json"])
        if not hit_spans:
            raise ValueError("Aggregated regex candidate has no stored hit positions")

        prepared_text, input_strategy, merged_sections = build_prepared_input(
            source_text,
            hit_spans=hit_spans,
            reference_normalized=row["reference_normalized"],
            reference_type=row["reference_type"],
        )

        input_path.write_text(prepared_text, encoding="utf-8", newline="\n")
        prepared_chars = len(prepared_text)

    except Exception as exc:
        preparation_status = "error"
        preparation_error = f"{type(exc).__name__}: {exc}"

    prepared_rows.append(
        {
            "job_id": row["job_id"],
            "reference_type": row["reference_type"],
            "reference_normalized": row["reference_normalized"],
            "record_id": row["record_id"],
            "affId": row["affId"],
            "procedureId": row["procedureId"],
            "case_number_clean": row["case_number_clean"],
            "source_text_path": str(source_path or ""),
            "prepared_input_path": str(input_path),
            "input_strategy": input_strategy,
            "reference_count": row["reference_count"],
            "original_document_chars": original_chars,
            "prepared_input_chars": prepared_chars,
            "prepared_sections_json": json.dumps(merged_sections),
            "preparation_status": preparation_status,
            "preparation_error": preparation_error,
        }
    )

input_manifest = pd.DataFrame(prepared_rows)
input_manifest.to_csv(INPUT_MANIFEST_CSV, index=False, encoding="utf-8")

print(f"Prepared inputs: {(input_manifest['preparation_status'] == 'ready').sum():,}")
print(f"Preparation errors: {(input_manifest['preparation_status'] != 'ready').sum():,}")
print("Input manifest:", INPUT_MANIFEST_CSV)

if (input_manifest["preparation_status"] != "ready").any():
    display(
        input_manifest.loc[
            input_manifest["preparation_status"] != "ready",
            ["job_id", "reference_normalized", "preparation_error"],
        ].head(30)
    )

## 4. Classification taxonomy

The first decision is whether the present Court proceeding is a **direct challenge to the identified Commission decision**.

`is_direct_appeal = true` means that the Commission decision itself is the measure directly challenged in this Court proceeding.

This deliberately does **not** include a later Court of Justice proceeding that directly appeals a General Court judgment, even when that General Court judgment concerned the same Commission decision. Such broader appeal-chain relationships can be added in a later step.

When the Commission decision is not directly challenged, one overall citation role is assigned:

- `legal_substantive` — the decision is used for legal reasoning, precedent, comparison, interpretation, or substantive argument;
- `procedural_evidentiary` — the decision is discussed mainly in relation to procedure, investigation, evidence, access, admissibility, or another procedural/evidentiary point;
- `background_historical` — the decision mainly forms part of the factual, institutional, chronological, or related-case background;
- `remedy_fine_penalty` — the decision is discussed mainly in relation to fines, penalties, remedies, sanctions, or their calculation;
- `unclear` — the supplied text does not support a reliable overall classification.

The classification is made for the **candidate as a whole**, using all supplied mentions together. It is not a majority vote over individual regex hits.

In [ ]:
SYSTEM_MESSAGE = """
You are classifying relationships in a legal research dataset.

Use only the supplied Court-document text. Do not use facts from memory.
The target Commission identifier has already been detected by an earlier regex step.
Your task is to classify its relationship to the present Court proceeding, not to
discover additional identifiers.

Return only JSON matching the requested schema.
""".strip()

CITATION_ROLES = [
    "direct_appeal",
    "legal_substantive",
    "procedural_evidentiary",
    "background_historical",
    "remedy_fine_penalty",
    "unclear",
]

RESPONSE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "is_direct_appeal": {
            "type": ["boolean", "null"],
            "description": (
                "True if this Court proceeding directly challenges the target "
                "Commission decision; false if it is only cited; null if unclear."
            ),
        },
        "citation_role": {
            "type": "string",
            "enum": CITATION_ROLES,
        },
        "reason": {
            "type": "string",
        },
        "evidence": {
            "type": "string",
        },
    },
    "required": [
        "is_direct_appeal",
        "citation_role",
        "reason",
        "evidence",
    ],
    "additionalProperties": False,
}


def shared_classification_instructions() -> str:
    return """
First determine whether the PRESENT COURT PROCEEDING directly challenges the
TARGET COMMISSION DECISION.

Use is_direct_appeal = true only when the Commission decision itself is the
measure directly challenged in this Court proceeding. Typical strong evidence
includes language such as an action/application for annulment of the Commission
decision, a request to annul that decision, or a direct challenge to the fine or
obligations imposed by that decision.

Use is_direct_appeal = false when the Commission decision is mentioned only as
background, precedent, comparison, procedural history, evidence, a related
matter, or another citation.

Important: if the present document concerns an appeal against a General Court
judgment, and that General Court judgment had reviewed the Commission decision,
do NOT call the present Court of Justice proceeding a direct appeal of the
Commission decision merely because the Commission decision appears in the
history. The direct object of that later appeal is the lower-court judgment.

If is_direct_appeal is true, citation_role must be "direct_appeal".

If is_direct_appeal is false, choose the ONE role that best describes how the
target Commission decision functions in the Court document overall:
- legal_substantive: legal reasoning, precedent, comparison, interpretation, or substantive argument;
- procedural_evidentiary: procedure, investigation, evidence, access, admissibility, or another procedural/evidentiary point;
- background_historical: factual, institutional, chronological, or related-case background;
- remedy_fine_penalty: fines, sanctions, remedies, penalties, or their calculation;
- unclear: the supplied text does not support a reliable overall classification.

Do not classify individual mentions separately. Consider all supplied mentions
together. A highly diagnostic passage can be more informative than many incidental
mentions, so do not use simple majority voting over occurrences.

If the direct-appeal relationship itself cannot be determined reliably from the
supplied text, set is_direct_appeal to null and citation_role to "unclear".

Keep reason concise. Evidence should quote or closely reproduce only the short
piece of supplied text most useful for the classification.
""".strip()


def build_case_number_prompt(
    *,
    reference_normalized: str,
    court_case: str,
    document_text: str,
) -> str:
    return f"""
TARGET TYPE: Commission competition case/procedure number
TARGET COMMISSION CASE NUMBER: {reference_normalized}
PRESENT COURT CASE: {court_case or '[not available]'}

Commission case numbers can identify the administrative competition proceeding
associated with a Commission decision. Determine from the supplied Court text
whether the PRESENT Court proceeding directly challenges the Commission decision
associated with this specific Commission case number.

{shared_classification_instructions()}

COURT DOCUMENT INPUT
--------------------
{document_text}
""".strip()


def build_oj_reference_prompt(
    *,
    reference_normalized: str,
    court_case: str,
    document_text: str,
) -> str:
    return f"""
TARGET TYPE: Published Commission decision reference
TARGET COMMISSION DECISION REFERENCE: {reference_normalized}
PRESENT COURT CASE: {court_case or '[not available]'}

The target is a published identifier for a Commission decision. Determine from
the supplied Court text whether the PRESENT Court proceeding directly challenges
this specific Commission decision.

{shared_classification_instructions()}

COURT DOCUMENT INPUT
--------------------
{document_text}
""".strip()

## 5. Resumable classification

Case-number candidates and published decision-reference candidates are classified in two separate runs with their own progress bars. They share the same checkpoint file and classification schema, but use different prompts.

Each completed attempt is appended immediately to the JSONL file. On the next `update` run, jobs whose latest record is successful are skipped.

In [ ]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_latest_checkpoint(path: Path) -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}
    if not path.exists():
        return latest

    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(f"Skipping malformed checkpoint line {line_number}")
                continue

            job_id = str(record.get("job_id", "")).strip()
            if job_id:
                latest[job_id] = record

    return latest


async def call_vllm(
    client: httpx.AsyncClient,
    prompt: str,
) -> dict[str, str]:
    endpoint = f"{VLLM_BASE_URL.rstrip('/')}/chat/completions"
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": prompt},
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "structured_outputs": {"json": RESPONSE_JSON_SCHEMA},
    }
    headers = {"Authorization": f"Bearer {VLLM_API_KEY}"}

    response = await client.post(endpoint, json=payload, headers=headers)
    response.raise_for_status()

    body = response.json()
    choices = body.get("choices") or []
    if not choices:
        raise ValueError("vLLM response contains no choices")

    content = choices[0].get("message", {}).get("content")
    if not content:
        raise ValueError("vLLM response contains no message content")

    return {
        "content": str(content),
        "finish_reason": str(choices[0].get("finish_reason") or ""),
    }


async def classify_one(
    row: dict[str, Any],
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    base_record = {
        "job_id": row["job_id"],
        "record_id": row["record_id"],
        "affId": row["affId"],
        "procedureId": row["procedureId"],
        "case_number_clean": row["case_number_clean"],
        "reference_type": row["reference_type"],
        "reference_normalized": row["reference_normalized"],
        "reference_count": row["reference_count"],
        "prepared_input_path": row["prepared_input_path"],
        "input_strategy": row["input_strategy"],
        "original_document_chars": row["original_document_chars"],
        "prepared_input_chars": row["prepared_input_chars"],
        "model": MODEL,
    }

    if row["preparation_status"] != "ready":
        return {
            **base_record,
            "classification_status": "error",
            "classification_error": row["preparation_error"],
            "classified_at": utc_now_iso(),
            "finish_reason": "",
            "raw_response": "",
            "parsed_response": None,
        }

    try:
        document_text = Path(row["prepared_input_path"]).read_text(
            encoding="utf-8",
            errors="replace",
        )
    except Exception as exc:
        return {
            **base_record,
            "classification_status": "error",
            "classification_error": f"{type(exc).__name__}: {exc}",
            "classified_at": utc_now_iso(),
            "finish_reason": "",
            "raw_response": "",
            "parsed_response": None,
        }

    if row["reference_type"] == "decision_case_number":
        prompt = build_case_number_prompt(
            reference_normalized=row["reference_normalized"],
            court_case=row["case_number_clean"],
            document_text=document_text,
        )
    elif row["reference_type"] == "decision_oj_reference":
        prompt = build_oj_reference_prompt(
            reference_normalized=row["reference_normalized"],
            court_case=row["case_number_clean"],
            document_text=document_text,
        )
    else:
        return {
            **base_record,
            "classification_status": "error",
            "classification_error": f"Unsupported reference type: {row['reference_type']}",
            "classified_at": utc_now_iso(),
            "finish_reason": "",
            "raw_response": "",
            "parsed_response": None,
        }

    last_error = ""
    async with semaphore:
        for attempt in range(1, MAX_RETRIES + 2):
            try:
                result = await call_vllm(client, prompt)
                parsed = json.loads(result["content"])

                if parsed.get("is_direct_appeal") is True:
                    parsed["citation_role"] = "direct_appeal"
                elif parsed.get("is_direct_appeal") is None:
                    parsed["citation_role"] = "unclear"
                elif parsed.get("citation_role") == "direct_appeal":
                    raise ValueError(
                        "citation_role cannot be direct_appeal when "
                        "is_direct_appeal is false"
                    )

                return {
                    **base_record,
                    "classification_status": "success",
                    "classification_error": "",
                    "classified_at": utc_now_iso(),
                    "attempt": attempt,
                    "finish_reason": result["finish_reason"],
                    "raw_response": result["content"],
                    "parsed_response": parsed,
                }

            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"

    return {
        **base_record,
        "classification_status": "error",
        "classification_error": last_error,
        "classified_at": utc_now_iso(),
        "attempt": MAX_RETRIES + 1,
        "finish_reason": "",
        "raw_response": "",
        "parsed_response": None,
    }


async def run_job_group(
    group_df: pd.DataFrame,
    *,
    description: str,
) -> list[dict[str, Any]]:
    if group_df.empty:
        print(f"{description}: nothing to run")
        return []

    semaphore = asyncio.Semaphore(MAX_WORKERS)
    timeout = httpx.Timeout(REQUEST_TIMEOUT_SECONDS)

    async with httpx.AsyncClient(timeout=timeout) as client:
        tasks = [
            asyncio.create_task(
                classify_one(row, client, semaphore)
            )
            for row in group_df.to_dict("records")
        ]

        results: list[dict[str, Any]] = []
        for task in tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc=description,
        ):
            record = await task
            append_jsonl(CHECKPOINT_JSONL, record)
            results.append(record)

    return results

In [ ]:
# Join the prepared-input metadata onto the aggregated regex jobs.
classification_jobs = jobs_df.merge(
    input_manifest,
    on=[
        "job_id",
        "reference_type",
        "reference_normalized",
        "record_id",
        "affId",
        "procedureId",
        "case_number_clean",
        "reference_count",
    ],
    how="left",
    validate="one_to_one",
)

if RUN_MODE == "redo_all" and CHECKPOINT_JSONL.exists():
    CHECKPOINT_JSONL.unlink()

latest_before_run = load_latest_checkpoint(CHECKPOINT_JSONL)

def needs_classification(job_id: str) -> bool:
    if RUN_MODE == "rebuild_excel":
        return False

    previous = latest_before_run.get(str(job_id))
    if previous is None:
        return True

    return previous.get("classification_status") != "success"

pending = classification_jobs.loc[
    classification_jobs["job_id"].map(needs_classification)
].copy()

pending_case_numbers = pending.loc[
    pending["reference_type"].eq("decision_case_number")
].copy()

pending_oj_references = pending.loc[
    pending["reference_type"].eq("decision_oj_reference")
].copy()

print(f"Already successful: {len(classification_jobs) - len(pending):,}")
print(f"Pending Commission case numbers: {len(pending_case_numbers):,}")
print(f"Pending published decision references: {len(pending_oj_references):,}")

In [ ]:
# Run the two identifier families separately.
# In Jupyter, top-level await is supported.

if RUN_MODE != "rebuild_excel":
    _case_number_results = await run_job_group(
        pending_case_numbers,
        description="Classifying Commission case numbers",
    )

    _oj_reference_results = await run_job_group(
        pending_oj_references,
        description="Classifying decision references",
    )

## 6. Assemble the candidate-level classification table

The final table preserves the grouped regex candidate from notebook 01 and adds the latest classification result for that same candidate. This keeps the output at the same document × Commission-identifier level used throughout the notebook.

In [ ]:
latest = load_latest_checkpoint(CHECKPOINT_JSONL)

classification_rows: list[dict[str, Any]] = []

for job_id, record in latest.items():
    parsed = record.get("parsed_response")
    parsed = parsed if isinstance(parsed, dict) else {}

    direct_value = parsed.get("is_direct_appeal")
    if direct_value is True:
        direct_status = "direct_appeal"
    elif direct_value is False:
        direct_status = "citation_only"
    else:
        direct_status = "unclear"

    classification_rows.append(
        {
            "job_id": job_id,
            "classification_status": record.get("classification_status", ""),
            "classification_error": record.get("classification_error", ""),
            "is_direct_appeal": direct_value,
            "direct_appeal_status": direct_status,
            "citation_role": parsed.get("citation_role", ""),
            "classification_reason": parsed.get("reason", ""),
            "classification_evidence": parsed.get("evidence", ""),
            "model": record.get("model", ""),
            "classified_at": record.get("classified_at", ""),
        }
    )

latest_df = pd.DataFrame(classification_rows)

final = jobs_df.merge(
    input_manifest[
        [
            "job_id",
            "prepared_input_path",
            "input_strategy",
            "original_document_chars",
            "prepared_input_chars",
            "preparation_status",
            "preparation_error",
        ]
    ],
    on="job_id",
    how="left",
    validate="one_to_one",
)

if len(latest_df):
    final = final.merge(
        latest_df,
        on="job_id",
        how="left",
        validate="one_to_one",
    )
else:
    for col in [
        "classification_status",
        "classification_error",
        "is_direct_appeal",
        "direct_appeal_status",
        "citation_role",
        "classification_reason",
        "classification_evidence",
        "model",
        "classified_at",
    ]:
        final[col] = ""

final["classification_status"] = final["classification_status"].fillna("not_run")

summary = (
    final.groupby(
        ["reference_type", "direct_appeal_status", "citation_role"],
        dropna=False,
    )
    .size()
    .reset_index(name="rows")
    .sort_values(
        ["reference_type", "direct_appeal_status", "citation_role"],
        kind="stable",
    )
)

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    final.to_excel(writer, sheet_name="classifications", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)

    for sheet_name in ("classifications", "summary"):
        ws = writer.sheets[sheet_name]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

print("Saved:", OUTPUT_XLSX)
print()
print("Classification status:")
display(
    final["classification_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="rows")
)
print()
display(summary)

## 7. Interpretation

The output remains at the same aggregated level as notebook 01: one row represents one normalized Commission identifier in one Court document, regardless of how many times that identifier appeared.

The stored prepared-input path makes it possible to inspect exactly what text supported each classification. The JSONL file is only used to resume the computational run; the Excel workbook is the consolidated candidate-level output used downstream.